# 01 — Data Cleaning & Merging
**Starbucks Customer Segmentation**

This notebook loads the three raw datasets, cleans them individually, and merges them into a single unified dataframe for analysis.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Load Raw Datasets

In [ ]:
# Load the three datasets
portfolio = pd.read_json('../data/portfolio.json', orient='records', lines=True)
profile   = pd.read_json('../data/profile.json',   orient='records', lines=True)
transcript= pd.read_json('../data/transcript.json',orient='records', lines=True)

print(f'Portfolio shape : {portfolio.shape}')
print(f'Profile shape   : {profile.shape}')
print(f'Transcript shape: {transcript.shape}')

In [ ]:
# Quick peek at each dataset
print('--- Portfolio ---')
display(portfolio.head(3))
print('--- Profile ---')
display(profile.head(3))
print('--- Transcript ---')
display(transcript.head(3))

## 3. Clean Portfolio

In [ ]:
print('Portfolio columns:', portfolio.columns.tolist())
print('Null values:\n', portfolio.isnull().sum())
portfolio.dtypes

In [ ]:
# Expand 'channels' list column into binary dummy columns
portfolio_clean = portfolio.copy()
portfolio_clean = portfolio_clean.join(
    pd.get_dummies(portfolio_clean['channels'].apply(pd.Series).stack()
                   .reset_index(level=1, drop=True)).groupby(level=0).sum()
)
portfolio_clean.drop(columns=['channels'], inplace=True)
portfolio_clean.rename(columns={'id': 'offer_id'}, inplace=True)

print('Cleaned portfolio shape:', portfolio_clean.shape)
display(portfolio_clean.head())

## 4. Clean Profile

In [ ]:
print('Profile columns:', profile.columns.tolist())
print('\nNull values:\n', profile.isnull().sum())
print('\nAge unique extremes:', profile['age'].min(), profile['age'].max())

In [ ]:
profile_clean = profile.copy()

# Age 118 is a placeholder for missing age — remove those rows
profile_clean = profile_clean[profile_clean['age'] != 118]

# Drop rows with missing income or gender
profile_clean.dropna(subset=['income', 'gender'], inplace=True)

# Convert became_member_on to datetime and extract membership year
profile_clean['became_member_on'] = pd.to_datetime(profile_clean['became_member_on'], format='%Y%m%d')
profile_clean['membership_year'] = profile_clean['became_member_on'].dt.year
profile_clean['membership_days'] = (pd.Timestamp('2018-08-01') - profile_clean['became_member_on']).dt.days

profile_clean.rename(columns={'id': 'customer_id'}, inplace=True)

print('Cleaned profile shape:', profile_clean.shape)
display(profile_clean.head())

## 5. Clean Transcript

In [ ]:
print('Transcript columns:', transcript.columns.tolist())
print('\nEvent types:', transcript['event'].value_counts())

In [ ]:
transcript_clean = transcript.copy()
transcript_clean.rename(columns={'person': 'customer_id'}, inplace=True)

# Expand the 'value' dict column
transcript_clean = transcript_clean.join(transcript_clean['value'].apply(pd.Series))
transcript_clean.drop(columns=['value'], inplace=True)

# Normalize offer id columns
if 'offer id' in transcript_clean.columns:
    transcript_clean.rename(columns={'offer id': 'offer_id'}, inplace=True)
if 'offer_id' not in transcript_clean.columns and 'offer id' not in transcript_clean.columns:
    transcript_clean['offer_id'] = np.nan

print('Cleaned transcript shape:', transcript_clean.shape)
display(transcript_clean.head())

## 6. Merge All Datasets

In [ ]:
# Merge transcript with profile
df = transcript_clean.merge(profile_clean, on='customer_id', how='inner')

# Merge with portfolio on offer_id (only for offer-related events)
df = df.merge(portfolio_clean, on='offer_id', how='left')

print('Final merged dataframe shape:', df.shape)
display(df.head())

## 7. Save Cleaned Data

In [ ]:
df.to_csv('../data/cleaned_data.csv', index=False)
print('Cleaned data saved to ../data/cleaned_data.csv')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())